# Build the RAG Pipeline

In this lab I build a complete Retrieval-Augmented Generation (RAG) system.

The pipeline consists of:

1. Indexing documents into a Chroma vector database
2. Retrieving the most relevant passages for a question
3. Augmenting the prompt with retrieved context
4. Generating a grounded answer using an LLM

The goal is to answer only from the knowledge base, provide citations, and refuse to answer when the information is not available.

In [2]:
!pip install chromadb

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
    --------------------------------------- 0.5/23.5 MB 3.6 MB/s eta 0:00:07
   ----- ---------------------------------- 3.1/23.5 MB 11.1 MB/s eta 0:00:02
   -------- ------------------------------- 5.0/23.5 MB 9.6 MB/s eta 0:00:02
   -------------------------- ------------- 15.7/23.5 MB 23.3 MB/s eta 0:00:01
   ---------------------------- ----------- 16.8/23.5 MB 18.5 MB/s eta 0:00:01
   ------------------------------- -------- 18.4/23.5 MB 16.2 MB/s eta 0:00:01
   ------------------------------------ --- 21.2/23.5 MB 16.5 MB/s eta 0:00:01
   ---------------------------------------  23.1/23.5 MB 15.3 MB/s eta 0:00:01
   ---------------------------------------- 23.5/23.5 MB 14.5 MB/s  0:00:01
   ---------------------------------------- 0.0/4.6 MB ? eta -:--:--
   ------------------------------------ --- 4.2/4.6 MB 21.5 MB/s eta 0:00:01
   ---------------------------------------- 4.6/4.6 MB 12.8 MB/s  0:00:00

   ---


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
## import json
import os

import chromadb
from sentence_transformers import SentenceTransformer

from google import genai

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Knowledge Base

The knowledge base contains passages with:

- id
- source
- text

Each passage is already a good chunk.

If these were full documents, we would normally split them into smaller chunks before embedding to improve retrieval quality.

In [4]:
with open(
    "knowledge_base.json",
    "r",
    encoding="utf-8"
) as f:

    knowledge_base = json.load(f)

print("Documents:", len(knowledge_base))

for item in knowledge_base:
    print(
        item["id"],
        "-",
        item["source"]
    )

Documents: 10
kb-01 - handbook.md
kb-02 - handbook.md
kb-03 - handbook.md
kb-04 - policy.md
kb-05 - policy.md
kb-06 - policy.md
kb-07 - it.md
kb-08 - it.md
kb-09 - it.md
kb-10 - facilities.md


## Load Embedding Model

I use the same embedding model for both indexing and querying.

Using the same embedding space is important because document vectors and query vectors must be directly comparable.

In [6]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6222.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


## Create Chroma Collection

Chroma stores document embeddings and allows semantic search.

Instead of manually comparing a query against every document with NumPy as in the previous lab, Chroma handles vector retrieval internally.

In [7]:
client = chromadb.Client()

collection = client.create_collection(
    name="knowledge_base"
)

print("Collection created.")

Collection created.


## Index Documents

Each document is embedded once and stored in Chroma together with:

- document text
- source metadata
- document id

This creates the retrieval index used later by the RAG pipeline.

In [8]:
for item in knowledge_base:

    embedding = embedding_model.encode(
        item["text"]
    ).tolist()

    collection.add(
        ids=[item["id"]],
        documents=[item["text"]],
        embeddings=[embedding],
        metadatas=[
            {
                "source": item["source"]
            }
        ]
    )

print(
    "Indexed",
    len(knowledge_base),
    "documents."
)

Indexed 10 documents.


In [9]:
results = collection.query(
    query_embeddings=[
        embedding_model.encode(
            "reset password"
        ).tolist()
    ],
    n_results=3
)

print(results)

{'ids': [['kb-07', 'kb-02', 'kb-05']], 'embeddings': None, 'documents': [["Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.", "To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.", 'To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'it.md'}, {'source': 'handbook.md'}, {'source': 'policy.md'}]], 'distances': [[0.46836280822753906, 1.4715800285339355, 1.5476773977279663]]}


## Retrieve Relevant Passages

Given a question, I retrieve the top 3 most relevant passages from Chroma.

The same embedding model is used for the query and the indexed documents.

The retrieved passages will later be inserted into the prompt for generation.

In [10]:
def retrieve(question, top_k=3):

    query_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    retrieved_docs = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    ids = results["ids"][0]

    for doc_id, doc_text, metadata in zip(
        ids,
        documents,
        metadatas
    ):

        retrieved_docs.append(
            {
                "id": doc_id,
                "source": metadata["source"],
                "text": doc_text
            }
        )

    return retrieved_docs

In [11]:
question = "How do I reset my password?"

results = retrieve(question)

for item in results:

    print("=" * 60)

    print("ID:", item["id"])
    print("Source:", item["source"])

    print()
    print(item["text"])

ID: kb-07
Source: it.md

Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.
ID: kb-02
Source: handbook.md

To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.
ID: kb-05
Source: policy.md

To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.


## Build the RAG Prompt

The retrieved passages are assembled into a grounded prompt.

The model is instructed to:

- answer only from the provided context
- cite the source of each fact
- say "I don't know" if the answer is not supported by the retrieved passages

This reduces hallucinations and makes the answer traceable to the knowledge base.

In [19]:
def build_prompt(question, retrieved_docs):

    context = ""

    for doc in retrieved_docs:

        context += (
            f"[Source: {doc['source']}]\n"
            f"{doc['text']}\n\n"
        )

    prompt = f"""
You are a helpful assistant.

Answer ONLY from the provided context.

If the answer is not explicitly contained in the context,
reply exactly:

I don't know.

Do not use outside knowledge.
Do not guess.
Do not infer missing information.

For every fact you use, include the source.

Context:

{context}

Question:
{question}
"""

    return prompt

In [20]:
question = "How do I reset my password?"

retrieved_docs = retrieve(question)

prompt = build_prompt(
    question,
    retrieved_docs
)

print(prompt)


You are a helpful assistant.

Answer ONLY from the provided context.

If the answer is not explicitly contained in the context,
reply exactly:

I don't know.

Do not use outside knowledge.
Do not guess.
Do not infer missing information.

For every fact you use, include the source.

Context:

[Source: it.md]
Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.

[Source: handbook.md]
To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.

[Source: policy.md]
To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.



Question:
How do I reset my password?



## Generate Answers with Ollama

The final step is **generation**.

The model receives:

- The user's question
- The retrieved passages
- Instructions to answer only from the provided context

The answer is generated by a **local Ollama model** instead of a hosted API.

### Benefits:

- No API cost
- No internet required
- Data stays on the local machine

If the information is not present in the retrieved passages, the model should answer:

> I don't know.

In [21]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"

print("Ollama ready.")

Ollama ready.


In [22]:
def generate_with_ollama(prompt):

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    data = response.json()

    return data["response"]

In [23]:
def ask_rag(question):

    retrieved_docs = retrieve(
        question,
        top_k=3
    )

    prompt = build_prompt(
        question,
        retrieved_docs
    )

    answer = generate_with_ollama(
        prompt
    )

    return {
        "question": question,
        "retrieved_docs": retrieved_docs,
        "answer": answer
    }

In [24]:
result = ask_rag(
    "How do I reset my password?"
)

print("QUESTION:")
print(result["question"])

print()

print("RETRIEVED SOURCES:")

for doc in result["retrieved_docs"]:

    print(
        f"- {doc['id']} ({doc['source']})"
    )

print()

print("ANSWER:")
print(result["answer"])

QUESTION:
How do I reset my password?

RETRIEVED SOURCES:
- kb-07 (it.md)
- kb-02 (handbook.md)
- kb-05 (policy.md)

ANSWER:
According to [it.md], you can reset your password by clicking 'Forgot password' on the login screen, which will send a reset link to your registered address that expires after one hour for security.


## Required Questions

The following questions are tested against the RAG pipeline:

1. How long do I have to get a full refund?

2. How do I reset my password?

3. What is the company's stock price today?

The third question is intentionally outside the knowledge base and should be declined.

In [25]:
questions = [
    "How long do I have to get a full refund?",
    "How do I reset my password?",
    "What is the company's stock price today?"
]

for question in questions:

    result = ask_rag(question)

    print("=" * 80)

    print("QUESTION:")
    print(question)

    print()

    print("RETRIEVED SOURCES:")

    for doc in result["retrieved_docs"]:

        print(
            f"- {doc['id']} ({doc['source']})"
        )

    print()

    print("ANSWER:")
    print(result["answer"])

    print()

QUESTION:
How long do I have to get a full refund?

RETRIEVED SOURCES:
- kb-04 (policy.md)
- kb-05 (policy.md)
- kb-06 (policy.md)

ANSWER:
You have 30 days after purchase to get a full refund, provided the item is unused and in its original packaging. (Source: policy.md)

QUESTION:
How do I reset my password?

RETRIEVED SOURCES:
- kb-07 (it.md)
- kb-02 (handbook.md)
- kb-05 (policy.md)

ANSWER:
According to [it.md], you can reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.

QUESTION:
What is the company's stock price today?

RETRIEVED SOURCES:
- kb-10 (facilities.md)
- kb-09 (it.md)
- kb-06 (policy.md)

ANSWER:
I don't know.



## Results and Reflection

The RAG pipeline successfully retrieved relevant passages from the knowledge base and generated grounded answers using only the retrieved context.

For the refund question, the system retrieved the refund policy passage and correctly answered that a full refund is available within 30 days.

For the password reset question, the system retrieved the IT support passage and provided the correct password reset instructions.

For the stock price question, no relevant information existed in the knowledge base. The system correctly responded with "I don't know" instead of inventing an answer.

This demonstrates one of the main benefits of Retrieval-Augmented Generation (RAG): answers can be grounded in specific documents while reducing hallucinations through retrieval and source citations.